# Qorx 1.0.6 — provider comparison

Reproduce the three-question full-context versus Qorx check published in `docs/benchmarks/datacamp-provider-comparison-2026-07-22.md`. Run once per provider. The notebook downloads the checksum-verified static Qorx binary and never stores an API key.

This is a small functional fixture, not a production quality, cost, energy, or latency benchmark. The published capture used DataCamp's managed starter clients. This portable notebook uses `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` plus optional `QORX_PROVIDER_BASE_URL`; it does not publish DataCamp's internal proxy configuration.

In [ ]:
from pathlib import Path
import hashlib, json, math, os, platform, statistics, subprocess, tarfile, time, urllib.request

PROVIDER = os.environ.get('QORX_PROVIDER', 'openai')  # openai or anthropic
MODELS = {'openai': 'gpt-4o-mini', 'anthropic': 'claude-sonnet-4-6'}
MODEL = os.environ.get('QORX_PROVIDER_MODEL', MODELS[PROVIDER])
API_KEY_ENV = 'OPENAI_API_KEY' if PROVIDER == 'openai' else 'ANTHROPIC_API_KEY'
API_KEY = os.environ.get(API_KEY_ENV)
if not API_KEY:
    raise RuntimeError(f'Set {API_KEY_ENV}; never paste a key into the notebook.')
BASE_URL = os.environ.get('QORX_PROVIDER_BASE_URL') or None
WORK = Path.cwd() / 'qorx-provider-comparison'
WORK.mkdir(exist_ok=True)
print({'provider': PROVIDER, 'model': MODEL, 'platform': platform.platform()})

In [ ]:
VERSION = '1.0.6'
TAG = f'v{VERSION}'
ASSET = f'qorx-{TAG}-linux-x64-static.tar.gz'
archive = WORK / ASSET
release_url = f'https://github.com/bbrainfuckk/qorx/releases/download/{TAG}/{ASSET}'
if not archive.exists():
    urllib.request.urlretrieve(release_url, archive)
checksum_file = WORK / f'{ASSET}.sha256'
if not checksum_file.exists():
    urllib.request.urlretrieve(f'{release_url}.sha256', checksum_file)
expected_archive_sha = checksum_file.read_text().split()[0].lower()
archive_sha = hashlib.sha256(archive.read_bytes()).hexdigest()
if archive_sha != expected_archive_sha:
    raise RuntimeError(f'asset checksum mismatch: {archive_sha} != {expected_archive_sha}')
release = WORK / 'release'
if not release.exists():
    release.mkdir()
    with tarfile.open(archive, 'r:gz') as package:
        destination = release.resolve()
        for member in package.getmembers():
            candidate = (destination / member.name).resolve()
            if candidate != destination and destination not in candidate.parents:
                raise RuntimeError(f'unsafe archive member: {member.name}')
        package.extractall(destination)
binary = next(path for path in release.rglob('qorx') if path.is_file())
binary.chmod(binary.stat().st_mode | 0o111)
home = WORK / f'home-{PROVIDER}'
home.mkdir(exist_ok=True)
qorx_env = os.environ.copy()
qorx_env['QORX_HOME'] = str(home)

def qorx(*args):
    result = subprocess.run([str(binary), *args], env=qorx_env, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError({'args': args, 'stdout': result.stdout, 'stderr': result.stderr})
    return result

assert qorx('--version').stdout.strip() == 'qorx 1.0.6'
print({'archive_sha256': archive_sha, 'binary_sha256': hashlib.sha256(binary.read_bytes()).hexdigest()})

In [ ]:
corpus = WORK / 'corpus'
corpus.mkdir(exist_ok=True)
(corpus / 'operations.md').write_text('''# Operations
Emergency production overrides require approval from the Reliability Lead.
The override review window is exactly 27 minutes.
Every override must cite incident ticket QRX-441.
''')
(corpus / 'release.md').write_text('''# Release
The Atlas release marker is cobalt-heron-73.
The release owner is Mina Sol.
Rollback requires two maintainers.
''')
for i in range(8):
    (corpus / f'distractor-{i}.txt').write_text(
        ('Routine telemetry line with no emergency, release, customer, or policy authority ' + str(i) + '\n') * 160
    )
qorx('index', str(corpus))
full_context = '\n\n'.join(
    f'## {path.name}\n{path.read_text()}' for path in sorted(corpus.iterdir())
)
print({'chars': len(full_context), 'local_estimated_tokens': math.ceil(len(full_context) / 4)})

In [ ]:
system_prompt = (
    'Answer only from the supplied CONTEXT. If the context does not explicitly support every '
    'requested fact, reply exactly NOT_SUPPORTED. Otherwise answer in one short sentence. '
    'Do not use outside knowledge.'
)

if PROVIDER == 'openai':
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

    def call_model(context, query):
        started = time.perf_counter_ns()
        response = client.chat.completions.create(
            model=MODEL, max_tokens=80, temperature=0,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f'CONTEXT:\n{context}\n\nQUESTION:\n{query}'},
            ],
        )
        elapsed_ms = (time.perf_counter_ns() - started) / 1_000_000
        usage = response.usage
        details = getattr(usage, 'prompt_tokens_details', None)
        return {
            'answer': response.choices[0].message.content.strip(),
            'input_tokens': usage.prompt_tokens,
            'output_tokens': usage.completion_tokens,
            'cache_read_input_tokens': getattr(details, 'cached_tokens', 0) or 0,
            'elapsed_ms': elapsed_ms,
        }
else:
    from anthropic import Anthropic
    client = Anthropic(api_key=API_KEY, base_url=BASE_URL)

    def call_model(context, query):
        started = time.perf_counter_ns()
        response = client.messages.create(
            model=MODEL, max_tokens=80, temperature=0, system=system_prompt,
            messages=[{'role': 'user', 'content': f'CONTEXT:\n{context}\n\nQUESTION:\n{query}'}],
        )
        elapsed_ms = (time.perf_counter_ns() - started) / 1_000_000
        usage = response.usage
        return {
            'answer': response.content[0].text.strip(),
            'input_tokens': usage.input_tokens,
            'output_tokens': usage.output_tokens,
            'cache_read_input_tokens': getattr(usage, 'cache_read_input_tokens', 0) or 0,
            'elapsed_ms': elapsed_ms,
        }

In [ ]:
questions = [
    {'id': 'supported_operations', 'query': 'What is the emergency production override review window, and who must approve it?', 'expected': ['27 minutes', 'Reliability Lead']},
    {'id': 'supported_release', 'query': 'What is the Atlas release marker, and who owns the release?', 'expected': ['cobalt-heron-73', 'Mina Sol']},
    {'id': 'unsupported_customer', 'query': "What is the customer's bank account number?", 'expected': ['NOT_SUPPORTED']},
]

def is_correct(question, answer):
    if question['id'] == 'unsupported_customer':
        return answer.strip() == 'NOT_SUPPORTED'
    lowered = answer.lower()
    return all(item.lower() in lowered for item in question['expected'])

rows = []
for question in questions:
    pack_started = time.perf_counter_ns()
    pack = json.loads(qorx('pack', question['query'], '--budget-tokens', '320').stdout)
    pack_ms = (time.perf_counter_ns() - pack_started) / 1_000_000
    for condition, context in [('full', full_context), ('qorx', pack['text'])]:
        observed = call_model(context, question['query'])
        rows.append({
            'question_id': question['id'], 'condition': condition,
            'context_chars': len(context), 'answer': observed['answer'],
            'correct': is_correct(question, observed['answer']),
            'provider_input_tokens': observed['input_tokens'],
            'provider_output_tokens': observed['output_tokens'],
            'cache_read_input_tokens': observed['cache_read_input_tokens'],
            'provider_elapsed_ms': observed['elapsed_ms'],
            'qorx_pack_ms': pack_ms if condition == 'qorx' else None,
            'qorx_indexed_tokens': pack['indexed_tokens'],
            'qorx_used_tokens': pack['used_tokens'] if condition == 'qorx' else None,
        })

full_rows = [row for row in rows if row['condition'] == 'full']
qorx_rows = [row for row in rows if row['condition'] == 'qorx']
full_input = sum(row['provider_input_tokens'] for row in full_rows)
qorx_input = sum(row['provider_input_tokens'] for row in qorx_rows)
result = {
    'schema': 'qorx.datacamp.provider-comparison.v1',
    'captured_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'provider': PROVIDER, 'model': MODEL, 'qorx_version': qorx('--version').stdout.strip(),
    'qorx_binary_sha256': hashlib.sha256(binary.read_bytes()).hexdigest(),
    'release_archive_sha256': archive_sha,
    'corpus_sha256': hashlib.sha256(full_context.encode()).hexdigest(),
    'questions': questions, 'rows': rows,
    'summary': {
        'full_provider_input_tokens': full_input,
        'qorx_provider_input_tokens': qorx_input,
        'provider_input_reduction_x': full_input / max(qorx_input, 1),
        'full_accuracy': sum(row['correct'] for row in full_rows) / len(full_rows),
        'qorx_accuracy': sum(row['correct'] for row in qorx_rows) / len(qorx_rows),
        'full_median_elapsed_ms': statistics.median(row['provider_elapsed_ms'] for row in full_rows),
        'qorx_median_elapsed_ms': statistics.median(row['provider_elapsed_ms'] for row in qorx_rows),
        'qorx_median_pack_ms': statistics.median(row['qorx_pack_ms'] for row in qorx_rows),
    },
}
result_path = WORK / f'qorx-{PROVIDER}-{MODEL}-comparison.json'
result_path.write_text(json.dumps(result, indent=2) + '\n')
print(json.dumps(result['summary'], indent=2))
print('saved:', result_path)